### Imports

In [301]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np



In [302]:
texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever",
    "The food was awful"
]



# 1 = positive, 0 = negative

labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0])

In [303]:
vocab_size = 1000
max_length = 5

tokenizer = Tokenizer(num_words = vocab_size, oov_token = "<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

X =pad_sequences(sequences, maxlen = max_length, padding ="post")

print("word Index: ")
print(tokenizer.word_index)

print("\nInput sequences: ")
print(X)

word Index: 
{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'love': 9, 'is': 10, 'amazing': 11, 'good': 12, 'excellent': 13, 'hate': 14, 'terrible': 15, 'boring': 16, 'worst': 17, 'ever': 18, 'the': 19, 'food': 20, 'was': 21, 'awful': 22}

Input sequences: 
[[ 3  9  2  4  0]
 [ 2  5 10 11  0]
 [ 6 12  7  0  0]
 [13  8  0  0  0]
 [ 3 14  2  4  0]
 [15  5  0  0  0]
 [ 6 16  8  0  0]
 [17  7 18  0  0]
 [19 20 21 22  0]]


In [304]:
embed_dim = 16
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(   # <---- token embedding converts each word to vectors 
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(  # <---- positional embedding allocates the position of each word in the sentence
            input_dim=max_length,
            output_dim=embed_dim
        )



    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb
    
    print()

In [305]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(  # <---- self attention layer for the model to understand the relationships
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="sigmoid"),  # <---- feed Forward network the output goes through FFNN
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()  # <---- stabalizes the values which helps the model to train better
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [306]:
# build the model
embed_dim = 64
num_heads = 8
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x) # <--- the output layer
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [307]:
model.compile(optimizer = "adam", loss = "binary_crossentropy", metrics = ["accuracy"])

model.summary()

Model: "functional_67"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_66 (InputLayer)     │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_33 │ (None, 5, 64)          │        64,320 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_33            │ (None, 5, 64)          │       137,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_33     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_101 (Dense)               │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,505 (787.13 KB)

 Trainable params: 201,505 (787.13 KB)

 Non-trainable params: 0 (0.00 B)

In [308]:
model.fit(X, labels, epochs = 20, batch_size = 2 , verbose = 1)

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1111 - loss: 1.3140    
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5556 - loss: 0.8089 
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7778 - loss: 0.5167 
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6667 - loss: 0.6307 
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7778 - loss: 0.5086     
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8889 - loss: 0.3675 
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.3109 
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.2041 
Epoch 9/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.1393 
Epoch 10/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0812 
Epoch 11/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0480 
Epoch 12/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.

In [309]:
test_sentences = [
    "I love the film",
    "This movie was awful",
    "the food was really terrible",
    "had the worst day of my life"
    # "The food was Excellent"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
I love the film -> 0.87943494
Prediction: Positive
This movie was awful -> 0.0042721005
Prediction: Negative
the food was really terrible -> 0.0001707563
Prediction: Negative
had the worst day of my life -> 0.035403613
Prediction: Negative


## Flow Diagram

### Input Sentence --> Tokenzation + Padding --> Token Embedding --> Positional Embedding --> Embedded Input (tooken + postional) --> Self Attention --> Feed Forward Neural Network --> Normalization of the output --> Model building --> Model Compile --> Model Fit --> Final Output


### The appropriate combination of embed_dim and num_head is 64 & 8.